# AQUA HORIZON - AI Flood Prediction for 725 Indian Districts
**Run this notebook top to bottom.** Every Python file of the project is run below, in workflow order, and prints its own output.
At the end, `04_final_report.py` collects the results of all files into one report, and the website is shown inside the notebook.

### How to start in Google Colab
1. Upload **`AquaHorizon_Colab.ipynb`** to Colab (File > Upload notebook).
2. Run the first code cell and choose **`AquaHorizon.zip`** when it asks for a file.
3. Run the remaining cells one after another (Runtime > Run all also works).

Tip: choose **Runtime > Change runtime type > GPU** for faster neural-network training. The code also works on a normal CPU.

### Workflow (which file feeds which)
| Step | File | Needs | Produces |
|---|---|---|---|
| 1 | `ml/01_preprocess.py` | raw flood records in `data/raw/` | `data/processed/district_year_dataset.csv`, `district_profiles.json` |
| 2 | (no separate balancing step) | step 1 | the dataset is never balanced as a whole: SMOTE runs inside each training split only, so test data stays real |
| 3 | `ml/data_loader.py`, `ml/models_hybrid.py`, `ml/ensemble.py` | step 1 | library files used by later steps; each has a self-test that prints output |
| 4 | `ml/02_train_models.py` | step 1 | XGBoost district model (`champion_classifier.joblib`, `severity_regressor.joblib`), `benchmark_metrics.json` |
| 5 | `ml/train_all_hybrid_models.py` (runs `train_01` to `train_05`) | steps 1, 3 | the 5 deep-learning models, `hybrid_benchmarks.json` |
| 6 | `ml/train_kfold_hybrid_models.py` | steps 1, 3 | 10-fold results of the 5 models, `kfold_district_predictions.csv` |
| 7 | `ml/train_combo_hybrid_models.py` | steps 1, 3 | the top-3 models and their 4 combinations, `combo_hybrid_benchmarks.json`, `combo_district_predictions.csv` |
| 8 | `ml/test_10fold_irs.py`, `ml/test_satellite_learning.py` | step 1 | Inundation Risk Score and flooded-area results |
| 9 | `ml/download_sen1floods11_india.py`, `ml/image_pipeline_india.py` | internet | satellite image results (CLAHE, GAN, 10-fold) |
| 10 | `ml/03_export_web_data.py` | steps 4, 6, 7 + live rain forecast | `web/data/*.json` (what the website shows) |
| 11 | `predict.py` | steps 1, 4 | flood-probability report for one district |
| 12 | `ml/04_final_report.py` | everything above | one consolidated report |
| 13 | `run_app.py` | step 10 | the website |

## Setup: load the project

In [ ]:
import os, sys, zipfile

IN_COLAB = 'google.colab' in sys.modules
PROJECT = '/content/AquaHorizon' if IN_COLAB else os.getcwd()

if IN_COLAB and not os.path.exists(PROJECT):
    from google.colab import files
    print('Choose AquaHorizon.zip from your computer ...')
    uploaded = files.upload()
    for name in uploaded:
        if name.lower().endswith('.zip'):
            zipfile.ZipFile(name).extractall('/content')

os.chdir(PROJECT)
print('Working folder:', os.getcwd())
print('Contents:', sorted(os.listdir('.')))

In [ ]:
# Install only the packages Colab does not already have, then show the hardware
import importlib.util, subprocess, sys
needed = {'tifffile': 'tifffile', 'imblearn': 'imbalanced-learn', 'xgboost': 'xgboost', 'cv2': 'opencv-python-headless', 'scipy': 'scipy'}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
print('Installed now:', missing or 'nothing (everything was already available)')
import torch
print('PyTorch', torch.__version__, '| GPU available:', torch.cuda.is_available())

## Step 1 - `ml/01_preprocess.py`: build the dataset
Reads the 1967-2023 India flood records and builds one row per **district and year** with the features the models learn from:
the district severity score **DFSI** `= (10*deaths + 2*injured + 1) / log10(population) * (ln(1+flooded area %) + 0.1) * ln(1+mean flood duration)`,
flood history over the previous 3, 5 and 10 years, drainage stress, and a rainfall feature.
 (about 4 sec)

In [ ]:
!python -u ml/01_preprocess.py

In [ ]:
import pandas as pd
df = pd.read_csv('data/processed/district_year_dataset.csv')
print(df.shape, '- flood years:', f"{df['flood_occurred'].mean()*100:.1f}%")

# Look at one district: change the name to see any other district
DISTRICT = 'Vellore'
d = df[df['Dist_Name'].str.lower() == DISTRICT.lower()]
print(f"{DISTRICT}: {int(d['flood_occurred'].sum())} flood years out of {len(d)} (1967-2023). Latest 5 years:")
d[['Dist_Name', 'year', 'flood_occurred', 'prior_3yr_events', 'prior_5yr_events', 'dfsi_score', 'dfsi_rank', 'rainfall_anomaly_pct']].tail(5).round(2)

## Step 3 - library files with self-tests
`data_loader.py` cuts the data into 5-year windows, `models_hybrid.py` defines the 5 neural networks, and `ensemble.py` averages several models.
Each prints a short self-test so you can see it working.
 (about 10 sec)

In [ ]:
!python -u ml/data_loader.py

In [ ]:
!python -u ml/models_hybrid.py

In [ ]:
!python -u ml/ensemble.py

## Step 4 - `ml/02_train_models.py`: XGBoost district model
Trains the district-level flood classifier and compares three ways of handling class imbalance (none, SMOTE, ADASYN) with
10-fold cross-validation and a split by year. The ADASYN fold models become the deployed ensemble, plus a flood-severity regressor.
 (about 23 sec)

In [ ]:
!python -u ml/02_train_models.py

## Step 5 - `ml/train_all_hybrid_models.py`: the 5 deep-learning models
Runs the five training files (`train_01_unet_convlstm.py`, `train_02_cnn_lstm.py`, `train_03_cnn_transformer.py`,
`train_04_resnet_bilstm.py`, `train_05_attention_unet_lstm.py`) one after the other. Each model reads 5 years of history for a district and
outputs a flood probability. Results go to `hybrid_benchmarks.json`.
 (about 6 min on a normal CPU; Colab's GPU runtime can be faster for the neural networks)

In [ ]:
!python -u ml/train_all_hybrid_models.py

## Step 6 - `ml/train_kfold_hybrid_models.py`: 10-fold validation of the 5 models
Trains each network 10 times on different parts of the data and reports accuracy, recall, precision, F1 and ROC-AUC (average and spread).
It also writes each model's flood probability for every district (`kfold_district_predictions.csv`), which the website uses.
 (about 11 min on a normal CPU; Colab's GPU runtime can be faster for the neural networks)

In [ ]:
!python -u ml/train_kfold_hybrid_models.py

## Step 7 - `ml/train_combo_hybrid_models.py`: combine the top 3 models
Takes the three best models (ResNet+BiLSTM, CNN+LSTM, CNN+Transformer) and builds 4 combinations by **averaging their probabilities**:
first two, last two, first + last, and all three. SMOTE is applied to training data only and the test data stays real,
so the single models and the combinations are compared fairly (80:20 split by year and 10-fold cross-validation).
 (about 7 min on a normal CPU; Colab's GPU runtime can be faster for the neural networks)

In [ ]:
!python -u ml/train_combo_hybrid_models.py

## Step 8 - `ml/test_10fold_irs.py` and `ml/test_satellite_learning.py`
The **Inundation Risk Score** is `IRS = P(flood) * ln(1 + predicted flooded area %)`: high only when a flood is likely and would cover a large area.
`test_satellite_learning.py` checks how well district features predict the flooded-area percentage.
 (about 18 sec)

In [ ]:
!python -u ml/test_10fold_irs.py

In [ ]:
!python -u ml/test_satellite_learning.py

## Step 9 - satellite images: `ml/download_sen1floods11_india.py` and `ml/image_pipeline_india.py`
Downloads real Sentinel-1 radar scenes of Indian floods (Sen1Floods11, public Google bucket; needs internet), then runs the image pipeline:
quality checks, **CLAHE** contrast enhancement, GAN-generated extra flood examples (training data only), and 10-fold cross-validation.
This is the slowest step.
 (about 13 min on a normal CPU; Colab's GPU runtime can be faster for the neural networks)

In [ ]:
!python -u ml/download_sen1floods11_india.py

In [ ]:
!python -u ml/image_pipeline_india.py

## Step 10 - `ml/03_export_web_data.py`: prepare the website data
Combines the trained models with a **live 7-day rain forecast** (Open-Meteo, needs internet) and soil wetness to compute a daily flood probability for every district and every model:
`daily = clip((0.04 + 0.30*annual probability) * basin factor * soil factor * rain factor, 2.5%, 92%)`.
Writes the files in `web/data/` that the website reads.
 (about 10 sec)

In [ ]:
!python -u ml/03_export_web_data.py

## Step 11 - `predict.py`: flood report for one district
Prints the flood probability for one district under four what-if rainfall scenarios. Change the district name to try another one.
 (about 2 sec)

In [ ]:
!python -u predict.py Patna

## Step 12 - `ml/04_final_report.py`: everything in one report
Collects the results of all the files above: which file produced what, the accuracy of all 9 flood models, the other benchmarks,
and the final output of the system - the districts with the highest flood risk today.
 (about 1 sec)

In [ ]:
!python -u ml/04_final_report.py

## Step 12b - `ml/verify_protocol.py`: proof of the training protocol
Retrains nothing. It shows, with real numbers, that (1) the CSV district data and the satellite images are handled by **separate** scripts and never mixed,
(2) the CSV data is split 80:20 and SMOTE is applied to the **training part only**, so the 20% test part keeps its real, unbalanced class ratio, in the holdout and in every one of the 10 folds,
(3) the satellite images are split 80:20 and the GAN adds synthetic flood chips to the **training part only**, and
(4) every saved metric (accuracy, recall, precision, F1, ROC-AUC, PR-AUC, IoU) of every model, with 10-fold mean and spread.
Needs Step 9 to have run first (it reads the downloaded satellite scenes). The text is also saved to `ml/models/protocol_proof.txt`. (about 1 min)

In [ ]:
!python -u ml/verify_protocol.py

### Step 12c - runtime proofs: run the real code and log what SMOTE, the GAN and the test data receive
These three files run the real training code with the slow training stubbed out and **log every call**. They prove with assertions that
(1) SMOTE only ever receives training rows (80% in the holdout, 90% in each CV fold) and never a test row,
(2) the 20% test part reaches the models and every metric byte-identical to the original real data, with its natural unbalanced class ratio, and
(3) the GAN is trained only on flood chips of the 80% image training part and its synthetic chips are added to that part only.
Nothing is retrained or saved. (a few seconds each)

In [ ]:
!python -u ml/proof_2_smote_rows.py

In [ ]:
!python -u ml/proof_3_test_rows.py

In [ ]:
!python -u ml/proof_4_gan_inputs.py

## Step 13 - `run_app.py`: the website
First an automatic check that every file the website needs exists, then the website itself is shown below (pick a model from the dropdown,
click a district, and open **Model Data** for all the tables).

In [ ]:
!python -u run_app.py --check

In [ ]:
%run run_app.py

---
**All 26 Python files were run above:**
`01_preprocess`, `data_loader`, `models_hybrid`, `ensemble`, `02_train_models`, `train_all_hybrid_models` (with `train_01` to `train_05`),
`train_kfold_hybrid_models`, `train_combo_hybrid_models`, `test_10fold_irs`, `test_satellite_learning`, `download_sen1floods11_india`,
`image_pipeline_india`, `03_export_web_data`, `predict`, `04_final_report`, `verify_protocol`, `proof_2_smote_rows`, `proof_3_test_rows`, `proof_4_gan_inputs`, `run_app`.